# Regularization — Linear Regression

আগের lesson-এ আমরা Normal Equation দিয়ে weights বের করেছি:

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

এই lesson-এর সমস্যা:

> **কখনো কখনো $X^TX$-এর inverse exist করে না, আর near-duplicate features থাকলে inverse/weights অস্বাভাবিকভাবে বড় হয়ে যেতে পারে।**

এই সমস্যা সামলানোর একটি technique হলো **Regularization**।

এই lesson-এর flow:

```text
Duplicate columns
      ↓
Singular matrix → inverse নেই

Near-duplicate columns
      ↓
Inverse অনেক বড়
      ↓
Weights অনেক বড় / unstable

Regularization
      ↓
X.T @ X + rI
      ↓
More stable inverse
      ↓
Controlled weights
```

Normal Equation-এর regularized form:

$$
\boxed{
w=(X^TX+rI)^{-1}X^Ty
}
$$

এই technique-টি **Ridge Regression**-এর ভিত্তি।

# 1. Problem — Duplicate Columns

ধরি feature matrix:

```python
X = [
    [4, 4, 4],
    [3, 5, 5],
    [5, 1, 1],
    [5, 4, 4],
    [7, 5, 5],
    [4, 5, 5],
]
```

এখানে:

- column 1 = `[4, 3, 5, 5, 7, 4]`
- column 2 = `[4, 5, 1, 4, 5, 5]`
- column 3 = `[4, 5, 1, 4, 5, 5]`

অর্থাৎ:

$$
\text{column 3}=\text{column 2}
$$

তাই একটি column আরেকটির exact duplicate।

এ ধরনের matrix-এ columns **linearly dependent** হয়ে যায়।

ফলে Gram Matrix:

$$
X^TX
$$

singular হতে পারে, এবং তার inverse পাওয়া যায় না।

### Linear combination — সহজ অর্থ

একটি column যদি অন্য column(s) দিয়ে তৈরি করা যায়, সেটি linear combination।

সবচেয়ে simple case:

$$
c=b
$$

অর্থাৎ `c` হলো `b`-এর duplicate।

In [ ]:
import numpy as np

X = np.array([
    [4, 4, 4],
    [3, 5, 5],
    [5, 1, 1],
    [5, 4, 4],
    [7, 5, 5],
    [4, 5, 5],
], dtype=float)

XTX = X.T.dot(X)

print("X.T @ X:")
print(XTX)


In [ ]:
# Try to invert the Gram Matrix

np.linalg.inv(XTX)


উপরের code-এ `LinAlgError: Singular matrix` আসবে।

কারণ:

$$
\text{column 2}=\text{column 3}
$$

তাই `X.T @ X` singular।

অর্থাৎ:

$$
\boxed{(X^TX)^{-1}\text{ exist করে না}}
$$

এজন্য Normal Equation-এর:

$$
(X^TX)^{-1}
$$

step-এ সমস্যা হয়।

# 2. Near-Duplicate Columns — আরও subtle Problem

Real data-তে columns সবসময় exact duplicate হয় না।

ধরি শেষ value-তে খুব ছোট difference আছে:

```python
5
5.00000001
```

তাহলে columns আর exact duplicate নয়।

Matrix technically invertible হতে পারে।

কিন্তু সমস্যা অন্য জায়গায়:

> **Inverse-এর values অস্বাভাবিকভাবে বড় হয়ে যেতে পারে।**

এর ফলে duplicated/near-duplicated features-এর weights-ও অনেক বড় হতে পারে, যেমন:

```text
+3,400,000
-3,400,000
```

দুইটা huge weight একে অপরকে প্রায় cancel করে prediction ঠিক রাখতে পারে।

অর্থাৎ model-এর weights **unstable** হয়ে যায়।

### Concept

```text
Duplicate features
      ↓
Model বুঝতে পারে না কোন feature-কে কত weight দেবে
      ↓
Weights অস্বাভাবিকভাবে বড় হতে পারে
```

এটাই numerical instability-এর মূল সমস্যা।

In [ ]:
# Almost-duplicate columns

X = np.array([
    [4, 4, 4],
    [3, 5, 5],
    [5, 1, 1],
    [5, 4, 4],
    [7, 5, 5],
    [4, 5, 5.00000001],
], dtype=float)

XTX = X.T.dot(X)
XTX_inv = np.linalg.inv(XTX)

print("Inverse of X.T @ X:")
print(XTX_inv)


খুব ছোট difference থাকার কারণে inverse পাওয়া গেছে।

কিন্তু inverse-এর values অনেক বড়।

এখানে গুরুত্বপূর্ণ idea:

> **Exact duplicate হলে inverse নেই; near-duplicate হলে inverse থাকতে পারে, কিন্তু সেটি numerically unstable হতে পারে।**

এখন এই instability control করার জন্য regularization ব্যবহার করব।

# 3. Regularization কীভাবে কাজ করে?

Normal Equation:

$$
w=(X^TX)^{-1}X^Ty
$$

Regularization-এ আমরা Gram Matrix-এর diagonal-এ একটি ছোট value যোগ করি:

$$
\boxed{
w=(X^TX+rI)^{-1}X^Ty
}
$$

এখানে:

- `r` = regularization parameter
- `I` = identity matrix

ধরি:

$$
r=0.01
$$

তাহলে:

$$
X^TX
\rightarrow
X^TX+0.01I
$$

### Identity Matrix

3 × 3 identity matrix:

$$
I=
\begin{bmatrix}
1&0&0\\
0&1&0\\
0&0&1
\end{bmatrix}
$$

তাই:

$$
0.01I=
\begin{bmatrix}
0.01&0&0\\
0&0.01&0\\
0&0&0.01
\end{bmatrix}
$$

এটা `XTX`-এর diagonal-এর সাথে যোগ হয়।

Python:

```python
0.01 * np.eye(3)
```

এবং:

```python
XTX + 0.01 * np.eye(3)
```

### মূল উদ্দেশ্য

এই ছোট diagonal addition matrix-কে আরও stable করে এবং inverse-এর huge values কমাতে সাহায্য করে।

ফলে weights-ও বেশি controlled থাকে।

In [ ]:
# Example: regularization on a small Gram Matrix

XTX = np.array([
    [1, 2, 2],
    [2, 1, 1],
    [2, 1, 1],
], dtype=float)

r = 0.01

XTX_reg = XTX + r * np.eye(XTX.shape[0])

print("Original XTX:")
print(XTX)

print("\nRegularized XTX:")
print(XTX_reg)


In [ ]:
# Now calculate the inverse

XTX_reg_inv = np.linalg.inv(XTX_reg)

print("Regularized inverse:")
print(XTX_reg_inv)


# 4. `r` কী?

`r` হলো regularization strength।

```python
r = 0
```

হলে:

$$
X^TX+rI=X^TX
$$

অর্থাৎ regularization নেই।

তাই আমরা আবার ordinary Linear Regression-এ ফিরে যাই।

---

`r` একটু বড় করলে:

$$
X^TX+rI
$$

এর diagonal আরও বাড়ে এবং weights আরও strongly controlled হয়।

কিন্তু `r` **অতিরিক্ত বড়** করলে model খুব বেশি constrained হয়ে যেতে পারে এবং performance খারাপ হতে পারে।

তাই:

$$
\boxed{r\text{ হলো একটি hyperparameter}}
$$

পরের lesson-এ আমরা বিভিন্ন `r` try করে best value খুঁজব।

# 5. Regularized Training Function

আগের training function ছিল:

```python
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]
```

Regularization-এ শুধু একটি নতুন line যোগ হয়েছে:

```python
XTX = XTX + r * np.eye(XTX.shape[0])
```

অর্থাৎ:

$$
X^TX
\rightarrow
X^TX+rI
$$

বাকি process একই।

In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    # Add bias column
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    # Gram Matrix
    XTX = X.T.dot(X)

    # Regularization
    XTX = XTX + r * np.eye(XTX.shape[0])

    # Normal Equation
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    # Bias and feature weights
    return w_full[0], w_full[1:]


# 6. Car Price Project-এ ব্যবহার

আমাদের previous project flow মনে করো:

```text
df_train
   ↓
prepare_X(df_train)
   ↓
X_train
   ↓
train_linear_regression_reg()
   ↓
w0, w
   ↓
X_val
   ↓
y_pred
   ↓
RMSE
```

Regularized training:

```python
X_train = prepare_X(df_train)

w0, w = train_linear_regression_reg(
    X_train,
    y_train,
    r=0.01
)

X_val = prepare_X(df_val)

y_pred = w0 + X_val.dot(w)

score = rmse(y_val, y_pred)
```

এখানে:

- `X_train` → model train করার features
- `y_train` → actual training targets
- `r=0.01` → regularization strength
- `w0, w` → trained parameters
- `y_pred` → validation predictions
- `score` → validation RMSE

In [ ]:
# Example usage in the car-price project

X_train = prepare_X(df_train)

w0, w = train_linear_regression_reg(
    X_train,
    y_train,
    r=0.01
)

X_val = prepare_X(df_val)

y_pred = w0 + X_val.dot(w)

score = rmse(y_val, y_pred)

print("Validation RMSE:", score)


# 7. কেন Regularization Model Improve করতে পারে?

ধরি দুইটি feature প্রায় একই information বহন করছে:

```text
Feature A → প্রায় একই
Feature B → প্রায় একই
```

Normal Equation model-কে দুই feature-এর weights এমনভাবে adjust করতে হতে পারে:

```text
A → +3,400,000
B → -3,400,000
```

দুটো প্রায় cancel হয়ে prediction ঠিক রাখতে পারে।

কিন্তু weights enormous এবং unstable।

Regularization model-কে খুব বড় weights নেওয়ার ব্যাপারে discourage করে।

ফলে weights সাধারণত আরও controlled হয়:

```text
A → ছোট weight
B → ছোট weight
```

অর্থাৎ regularization-এর মূল idea:

$$
\boxed{\text{weights-কে controlled রাখা}}
$$

এজন্যই নাম **regularization** — model-এর weights-কে regulate করা।

# 8. Important: Regularization ≠ Feature Removal

Regularization duplicate feature delete করছে না।

বরং model-এর calculation পরিবর্তন করছে:

### Without regularization

$$
(X^TX)^{-1}
$$

### With regularization

$$
\boxed{(X^TX+rI)^{-1}}
$$

অর্থাৎ feature matrix-এর original values একই থাকে, কিন্তু training-এর সময় Gram Matrix-এ diagonal adjustment যোগ হয়।

এতে numerical stability এবং weight control improve হয়।

# 9. Final Concept Map

## Problem

$$
w=(X^TX)^{-1}X^Ty
$$

কিন্তু:

```text
Duplicate columns
       ↓
X.T @ X singular
       ↓
Inverse নেই
```

অথবা:

```text
Near-duplicate columns
       ↓
Inverse খুব বড়
       ↓
Weights খুব বড় / unstable
```

## Solution

Gram Matrix-এর diagonal-এ `r` যোগ করি:

$$
\boxed{
X^TX\rightarrow X^TX+rI
}
$$

তারপর:

$$
\boxed{
w=(X^TX+rI)^{-1}X^Ty
}
$$

### Code-এর মূল পরিবর্তন

```python
XTX = X.T.dot(X)

XTX = XTX + r * np.eye(XTX.shape[0])

XTX_inv = np.linalg.inv(XTX)

w_full = XTX_inv.dot(X.T).dot(y)
```

### `r` সম্পর্কে

```text
r = 0
↓
ordinary Linear Regression

small r
↓
mild regularization

larger r
↓
stronger weight control

too large r
↓
model performance may get worse
```

`r` হলো **hyperparameter** এবং next lesson-এ এর best value tune করা হবে।

# Final Cheat Sheet

### Normal Equation

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

### Problem

`X.T @ X` singular বা numerically unstable হতে পারে যখন features duplicate/near-duplicate হয়।

### Regularization

$$
\boxed{
w=(X^TX+rI)^{-1}X^Ty
}
$$

### NumPy

```python
XTX = X.T.dot(X)

XTX = XTX + r * np.eye(XTX.shape[0])

XTX_inv = np.linalg.inv(XTX)

w_full = XTX_inv.dot(X.T).dot(y)
```

### One-line idea

> **Regularization Gram Matrix-এর diagonal-এ `r` যোগ করে weights-কে বেশি controlled এবং computation-কে বেশি stable করতে সাহায্য করে।**

**Ridge Regression** হলো এই type-এর regularization-এর standard নাম।